# Ejercicio práctico 2 en Google Colab: clima y violencia en Barranquilla

Cuaderno del módulo *Periodismo de precisión*. Repetimos con Python el ejercicio que hicimos en Excel y agregamos algunos análisis que en Excel son más difíciles.

La pregunta que guía el ejercicio es:

> ¿En Barranquilla hay más violencia los días más calientes?

El cuaderno tiene cuatro partes:

1. **Variable por variable:** un repaso rápido de cada variable, como en el ejercicio 1.
2. **Correlaciones:** la matriz de correlaciones, la pregunta guía y la comparación entre Pearson y Spearman.
3. **La tercera variable:** cómo el calendario explica buena parte de algunas correlaciones.
4. **La escala importa:** qué pasa cuando pasamos de días a meses.

**Cómo usar este cuaderno.** Ejecute las celdas en orden, de arriba hacia abajo, con el botón ▶ o con `Shift + Enter`. La primera celda lee la base desde el sitio del módulo.

**Fuente:** Trujillo, J. C. y Howley, P. (2018). *Dataset for weather and crime in Barranquilla, Colombia*. Mendeley Data. https://doi.org/10.17632/836wyfy5rn.1 (licencia CC BY 4.0). Los datos son **reales** y van de 2010 a 2016.

## 0. Preparar el cuaderno y cargar los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

URL = "https://raw.githubusercontent.com/nescolar/periodismo-de-precision/main/assets/datos/Dataset_clima_violencia_barranquilla.xlsx"
df = pd.read_excel(URL, sheet_name="Datos")
df.head()

### Reconocer la base

In [ ]:
print("Filas (días):", df.shape[0])
print("Variables:", df.shape[1])
print("Desde", df["fecha"].min().date(), "hasta", df["fecha"].max().date())
print("Valores faltantes:", df.isna().sum().sum())

In [ ]:
# Ordenamos los días de la semana y los tipos de día para que los gráficos salgan en orden
orden_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
orden_tipo = ["Entre semana", "Fin de semana", "Festivo"]

cuantitativas = ["violencia_interpersonal", "homicidios", "temp_max", "indice_calor",
                 "humedad", "lluvia", "viento", "luna_iluminada",
                 "comparendos_embriaguez", "matriculas_vehiculos"]

### Las herramientas del ejercicio 1

Volvemos a usar las dos funciones del ejercicio 1: una para variables **cualitativas** (frecuencias y barras) y otra para **cuantitativas** (medidas resumen, histograma y diagrama de caja).

In [ ]:
def describir_cualitativa(col, orden=None):
    conteo = df[col].value_counts()
    if orden is not None:
        conteo = conteo.reindex(orden)
    tabla = pd.DataFrame({"Frecuencia": conteo,
                          "Porcentaje": (conteo / conteo.sum() * 100).round(1)})
    display(tabla)
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(tabla) + 1.2))
    ax.barh(tabla.index.astype(str), tabla["Porcentaje"], color="#3b6ea5")
    for i, v in enumerate(tabla["Porcentaje"]):
        ax.text(v + 0.5, i, f"{v:.1f} %", va="center", fontsize=9)
    ax.set_xlim(0, tabla["Porcentaje"].max() * 1.18); ax.invert_yaxis()
    ax.set_xlabel("% de días"); ax.set_title(f"{col} (n = {conteo.sum()})", loc="left")
    plt.tight_layout(); plt.show()


def describir_cuantitativa(col):
    s = df[col]
    q1, q3 = s.quantile([0.25, 0.75])
    ric = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * ric, q3 + 1.5 * ric
    atipicos = s[(s < lim_inf) | (s > lim_sup)]
    resumen = pd.Series({
        "n": s.count(), "Media": s.mean(), "Mediana": s.median(),
        "Desviación estándar": s.std(), "Mínimo": s.min(),
        "Q1 (25 %)": q1, "Q3 (75 %)": q3, "RIC": ric, "Máximo": s.max(),
        "Asimetría": s.skew(),
        "Límite inferior (1,5 × RIC)": lim_inf, "Límite superior (1,5 × RIC)": lim_sup,
        "Valores atípicos": len(atipicos),
    }, name=col)
    display(resumen.to_frame())
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [3, 1]})
    a1.hist(s, bins=30, color="#3b6ea5", edgecolor="white")
    a1.axvline(s.mean(), color="#c0392b", linestyle="--", label=f"Media = {s.mean():.2f}")
    a1.axvline(s.median(), color="#222222", linestyle=":", label=f"Mediana = {s.median():.2f}")
    a1.set_title(f"Histograma de {col}", loc="left"); a1.set_ylabel("Días"); a1.legend(frameon=False)
    a2.boxplot(s, widths=0.5); a2.set_xticks([]); a2.set_title("Diagrama de caja", loc="left")
    plt.tight_layout(); plt.show()

## 1. Variable por variable

#### `tipo_dia`

Si el día fue festivo, de fin de semana o entre semana. Si un festivo cae en fin de semana, cuenta como festivo.

In [ ]:
describir_cualitativa("tipo_dia", orden=orden_tipo)

**Para comentar:** ¿Qué porcentaje de los días son festivos? ¿Cuántos días hay en cada grupo? Lo necesitaremos para comparar.

#### `violencia_interpersonal`

Casos de violencia interpersonal no letal en el día. El 86 % son riñas. **Es la variable que queremos explicar.**

In [ ]:
describir_cuantitativa("violencia_interpersonal")

**Para comentar:** ¿Cuántos casos hay en un día típico? ¿La distribución es simétrica o tiene una cola a la derecha? ¿Cuántos días son atípicos?

#### `homicidios`

Homicidios en el día.

In [ ]:
describir_cuantitativa("homicidios")

**Para comentar:** ¿Cuál es el valor más frecuente? ¿Qué forma tiene la distribución?

#### `temp_max`

Temperatura máxima del día, en °C. El IDEAM la registra en números enteros. **Es la otra variable de la pregunta guía.**

In [ ]:
describir_cuantitativa("temp_max")

**Para comentar:** ¿Cuánto varía la temperatura máxima de un día a otro? Compare la desviación estándar con la de la violencia. ¿Varía mucho o poco el calor en Barranquilla?

#### `indice_calor`

Índice de calor en °C: combina temperatura y humedad para medir la sensación de calor.

In [ ]:
describir_cuantitativa("indice_calor")

**Para comentar:** ¿Se parece su distribución a la de `temp_max`?

#### `humedad`

Humedad relativa promedio del día, en %.

In [ ]:
describir_cuantitativa("humedad")

**Para comentar:** ¿Cuál es la humedad típica? ¿Hay días atípicos?

#### `lluvia`

Precipitación total del día, en milímetros.

In [ ]:
describir_cuantitativa("lluvia")

**Para comentar:** ¿Por qué la mediana es 0? ¿Qué porcentaje de los días llueve? (Pista: calcule `(df['lluvia'] > 0).mean()`). ¿Qué medida de centro usaría para reportarla?

#### `viento`

Velocidad promedio del viento. La fuente no indica la unidad.

In [ ]:
describir_cuantitativa("viento")

**Para comentar:** Mire el máximo. ¿Es creíble? ¿Qué haría con ese dato antes de publicar?

#### `luna_iluminada`

Proporción de la Luna iluminada: 0 es luna nueva y 1, luna llena.

In [ ]:
describir_cuantitativa("luna_iluminada")

**Para comentar:** ¿Por qué el histograma tiene esta forma, con más días en los extremos? (Pista: piense en cómo cambia la luna a lo largo del mes).

#### `comparendos_embriaguez`

Comparendos por conducir embriagado. Los autores la usan como aproximación al consumo de alcohol.

In [ ]:
describir_cuantitativa("comparendos_embriaguez")

**Para comentar:** ¿Por qué la media es mucho mayor que la mediana?

#### `matriculas_vehiculos`

Matrículas de vehículos particulares. Los autores la usan como aproximación a la economía de la ciudad.

In [ ]:
describir_cuantitativa("matriculas_vehiculos")

**Para comentar:** ¿Hay muchos días con 0 matrículas? ¿Qué días cree que son?

### Los días más violentos

Un valor atípico no es necesariamente un error. Miremos cuáles son los días con más casos.

In [ ]:
df.nlargest(10, "violencia_interpersonal")[["fecha", "dia_semana", "tipo_dia", "violencia_interpersonal", "temp_max"]]

**Para comentar:** ¿Qué tienen en común estos días? ¿Reconoce alguna fecha? ¿Eran días especialmente calientes?

### La violencia a lo largo del tiempo

Como cada fila es un día, también podemos ver cómo cambia la violencia en el tiempo. Para que el gráfico se lea, promediamos por mes.

In [ ]:
mensual = df.set_index("fecha")["violencia_interpersonal"].resample("MS").mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mensual.index, mensual.values, color="#3b6ea5", linewidth=1.8)
ax.set_ylim(0, mensual.max() * 1.15)
ax.set_ylabel("Casos por día (promedio del mes)")
ax.set_title("Violencia interpersonal en Barranquilla, 2010-2016", loc="left")
plt.tight_layout(); plt.show()

df.groupby("anio")["violencia_interpersonal"].agg(["mean", "sum"]).rename(columns={"mean": "Promedio diario", "sum": "Total del año"})

**Para comentar:** ¿La violencia sube, baja o se mantiene entre 2010 y 2016? ¿Hay meses que se repiten como picos?

## 2. Correlaciones

Recordemos: el coeficiente de **Pearson** (*r*) mide qué tan fuerte es la relación **lineal** entre dos variables cuantitativas, y va de −1 a 1. Como guía práctica: |*r*| < 0,1 es prácticamente nula; entre 0,1 y 0,3, débil; entre 0,3 y 0,5, moderada; y mayor que 0,5, fuerte.

Incluimos también `festivo` y `fin_de_semana`. Como están codificadas con 1 y 0, su correlación con una variable cuantitativa indica si esa variable tiende a ser mayor los festivos o los fines de semana.

In [ ]:
def fuerza(r):
    r = abs(r)
    return "Prácticamente nula" if r < 0.1 else "Débil" if r < 0.3 else "Moderada" if r < 0.5 else "Fuerte"

variables = ["festivo", "fin_de_semana"] + cuantitativas
corr = df[variables].corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha="right", fontsize=9)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(v) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.7, label="r de Pearson")
ax.set_title("Matriz de correlaciones (Pearson)", loc="left")
plt.tight_layout(); plt.show()

### 2.1 Las correlaciones más fuertes

In [ ]:
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().rename("r").reset_index())
pares.columns = ["Variable 1", "Variable 2", "r"]
pares["Fuerza"] = pares["r"].apply(fuerza)
pares.reindex(pares["r"].abs().sort_values(ascending=False).index).head(12).reset_index(drop=True)

**Para comentar:** ¿Cuáles de estas relaciones son obvias por cómo se construyen las variables? ¿Cuáles le sorprenden?

### 2.2 ¿Con qué se relaciona la violencia?

In [ ]:
con_violencia = corr["violencia_interpersonal"].drop("violencia_interpersonal").sort_values()
display(con_violencia.to_frame("r con violencia").assign(Fuerza=con_violencia.apply(fuerza)))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(con_violencia.index, con_violencia.values,
        color=["#c0392b" if v > 0 else "#3b6ea5" for v in con_violencia.values])
ax.axvline(0, color="black", linewidth=0.8); ax.set_xlim(-1, 1); ax.set_xlabel("r de Pearson")
ax.set_title("Correlación de cada variable con la violencia interpersonal", loc="left")
plt.tight_layout(); plt.show()

### 2.3 La pregunta guía: calor y violencia

La temperatura máxima está registrada en números enteros, así que muchos puntos caerían uno encima de otro. Para verlos, les sumamos un pequeño desplazamiento aleatorio horizontal (*jitter*). Es solo para el gráfico: el cálculo de *r* usa los datos originales.

In [ ]:
def dispersion(x, y, jitter=0.0, datos=None):
    datos = df if datos is None else datos
    r = datos[x].corr(datos[y])
    m, b = np.polyfit(datos[x], datos[y], 1)
    rng = np.random.default_rng(1)
    xs_plot = datos[x] + rng.uniform(-jitter, jitter, len(datos))
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(xs_plot, datos[y], alpha=0.25, s=12, color="#3b6ea5")
    xs = np.linspace(datos[x].min(), datos[x].max(), 100)
    ax.plot(xs, m * xs + b, color="#c0392b", linewidth=2, label=f"Tendencia (r = {r:.2f})")
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(frameon=False)
    ax.set_title(f"{y} según {x}", loc="left")
    plt.tight_layout(); plt.show()
    print(f"r = {r:.3f} ({fuerza(r).lower()}). Por cada unidad adicional de {x}, "
          f"{y} cambia en promedio {m:+.2f}.")

dispersion("temp_max", "violencia_interpersonal", jitter=0.35)

In [ ]:
dispersion("indice_calor", "violencia_interpersonal")

In [ ]:
dispersion("temp_max", "homicidios", jitter=0.35)

Otra forma de verlo: el promedio de violencia para cada temperatura, junto con el número de días en que se calculó.

In [ ]:
por_temp = df.groupby("temp_max")["violencia_interpersonal"].agg(Días="count", Promedio="mean")
display(por_temp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(por_temp.index, por_temp["Promedio"], color="#3b6ea5")
for t, fila in por_temp.iterrows():
    ax.text(t, fila["Promedio"] + 0.2, f"n={int(fila['Días'])}", ha="center", fontsize=8)
ax.set_xlabel("Temperatura máxima (°C)"); ax.set_ylabel("Casos por día (promedio)")
ax.set_title("Violencia promedio según la temperatura máxima", loc="left")
plt.tight_layout(); plt.show()

**Para comentar:** Con estos resultados, ¿qué respondería a la pregunta guía? ¿Por qué no conviene sacar conclusiones de las barras de 28 °C o de 39 °C? Escriba un titular **honesto** de una línea.

### 2.4 Pearson y Spearman

La correlación de **Spearman** usa el orden de los datos en lugar de sus valores. Es más robusta ante valores atípicos y ante distribuciones muy asimétricas, como la de la lluvia o la de los comparendos. Si las dos se parecen, la conclusión es más sólida.

In [ ]:
otras = [v for v in variables if v != "violencia_interpersonal"]
comparacion = pd.DataFrame({
    "Pearson": df[otras].corrwith(df["violencia_interpersonal"], method="pearson"),
    "Spearman": df[otras].corrwith(df["violencia_interpersonal"], method="spearman"),
})
comparacion["Diferencia"] = comparacion["Spearman"] - comparacion["Pearson"]
comparacion

**Para comentar:** ¿En qué variables cambia más el resultado? ¿Alguna conclusión cambia de sentido?

## 3. ¿Una tercera variable?

La violencia se correlaciona **positivamente** con los comparendos por embriaguez (0,37) y **negativamente** con las matrículas de vehículos (−0,32). ¿Significa esto que el alcohol causa riñas y que comprar carro las evita? Antes de responder, miremos el calendario.

In [ ]:
por_dia = (df.groupby("dia_semana")[["violencia_interpersonal", "homicidios",
                                      "comparendos_embriaguez", "matriculas_vehiculos"]]
             .mean().reindex(orden_dias))
display(por_dia)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col, color in zip(axes, ["violencia_interpersonal", "comparendos_embriaguez", "matriculas_vehiculos"],
                          ["#c0392b", "#8e44ad", "#3b6ea5"]):
    ax.bar(por_dia.index.str[:3], por_dia[col], color=color)
    ax.set_title(col, loc="left", fontsize=10)
fig.suptitle("Promedio diario según el día de la semana", x=0.01, ha="left")
plt.tight_layout(); plt.show()

In [ ]:
df.groupby("tipo_dia")[["violencia_interpersonal", "homicidios",
                        "comparendos_embriaguez", "matriculas_vehiculos"]].mean().reindex(orden_tipo)

**Para comentar:** ¿Qué pasa los domingos con cada variable? ¿Qué mide realmente `matriculas_vehiculos` un domingo?

Ahora calculamos las mismas correlaciones **dentro** de cada tipo de día. Si la relación se mantiene dentro de cada grupo, no se debe solo al calendario. Si casi desaparece, el calendario era la tercera variable.

In [ ]:
filas = []
for tipo in orden_tipo:
    g = df[df["tipo_dia"] == tipo]
    filas.append({"tipo_dia": tipo, "Días": len(g),
                  "r comparendos-violencia": g["comparendos_embriaguez"].corr(g["violencia_interpersonal"]),
                  "r matrículas-violencia": g["matriculas_vehiculos"].corr(g["violencia_interpersonal"]),
                  "r temperatura-violencia": g["temp_max"].corr(g["violencia_interpersonal"])})
filas.append({"tipo_dia": "Todos los días", "Días": len(df),
              "r comparendos-violencia": df["comparendos_embriaguez"].corr(df["violencia_interpersonal"]),
              "r matrículas-violencia": df["matriculas_vehiculos"].corr(df["violencia_interpersonal"]),
              "r temperatura-violencia": df["temp_max"].corr(df["violencia_interpersonal"])})
pd.DataFrame(filas).set_index("tipo_dia").round(2)

In [ ]:
colores = {"Entre semana": "#3b6ea5", "Fin de semana": "#c0392b", "Festivo": "#e1a32a"}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for tipo in orden_tipo:
    g = df[df["tipo_dia"] == tipo]
    ax.scatter(g["comparendos_embriaguez"], g["violencia_interpersonal"], s=12, alpha=0.4,
               color=colores[tipo], label=f"{tipo} (n = {len(g)})")
ax.set_xlabel("comparendos_embriaguez"); ax.set_ylabel("violencia_interpersonal")
ax.set_title("Comparendos y violencia, coloreados por tipo de día", loc="left")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Para comentar:** Entre semana, ¿qué queda de la correlación entre comparendos y violencia? ¿Y entre matrículas y violencia? Explique con sus palabras por qué la correlación con todos los días era más fuerte.

Fíjese también en que, dentro del fin de semana, las matrículas siguen teniendo una correlación negativa con la violencia. La razón es otra vez el calendario, ahora a menor escala: el sábado se matricula (15 vehículos en promedio) y el domingo casi no (0,2), mientras que el domingo hay más violencia que el sábado. Si se calcula solo con los sábados o solo con los domingos no festivos, la correlación queda cerca de cero (0,06 y −0,09).

Ojo con el grupo de festivos: tiene solo 168 días, y algunos son muy atípicos (8 de diciembre, 25 de diciembre, 1 de enero). Una correlación calculada en un grupo pequeño con atípicos es poco estable.

## 4. La escala importa: de días a meses

Día por día, la lluvia y la violencia casi no se relacionan. Pero el 82 % de los días no llueve, así que la lluvia diaria es casi siempre cero. ¿Qué pasa si promediamos por mes?

In [ ]:
por_mes = (df.groupby(["anio", "num_mes"])
             .agg(violencia=("violencia_interpersonal", "mean"),
                  lluvia=("lluvia", "mean"),
                  temp_max=("temp_max", "mean"))
             .reset_index())
print("Meses:", len(por_mes))
print(f"Lluvia y violencia, por día: r = {df['lluvia'].corr(df['violencia_interpersonal']):.2f}")
print(f"Lluvia y violencia, por mes: r = {por_mes['lluvia'].corr(por_mes['violencia']):.2f}")
print(f"Temperatura y violencia, por día: r = {df['temp_max'].corr(df['violencia_interpersonal']):.2f}")
print(f"Temperatura y violencia, por mes: r = {por_mes['temp_max'].corr(por_mes['violencia']):.2f}")

In [ ]:
dispersion("lluvia", "violencia", datos=por_mes)

**Para comentar:** ¿Por qué cambia la correlación al cambiar de escala? Cuando una nota de prensa cite una correlación, ¿qué pregunta haría sobre la escala?

## 5. Conclusiones

Escriba en la celda de abajo (haga doble clic para editarla):

1. **La respuesta a la pregunta guía**, con el valor de *r*.
2. **Tres hallazgos** sobre qué se relaciona con la violencia en Barranquilla.
3. **Un ejemplo de tercera variable** que haya encontrado y cómo lo explicaría a un lector.
4. **Un titular honesto** y el gráfico que lo acompañaría.
5. **Una limitación** del análisis (piense en el periodo de los datos y en lo que miden de verdad los comparendos y las matrículas).

*Escriba aquí sus conclusiones.*

1.
2.
3.
4.
5.